# auto_corr_stock

> 对应代码：`EconometricsCode/code_in_notes/Chap.25/auto_corr_stock.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.25`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.25")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们载入个股行情数据，并只保留股票代码为 600519（贵州茅台）的样本。

In [ ]:
%%stata
use ../datasets/stock_price, clear
keep if stkcd==600519

数据本身没有现成的数值型时间序号，我们先按日期排序，再生成一个从 1 开始的序号 t，并把它设为时间变量，从而可以使用滞后算子 L.。

In [ ]:
%%stata
// 设定时间并计算收益率
sort date
gen t=_n
tsset t

我们计算对数收益率：收盘价取对数后做一阶差分，即 r_t = ln P_t − ln P_{t−1}，它在收益率较小时近似等于简单收益率，且具有良好的可加性。

In [ ]:
%%stata
gen rr=log(clsprc)-log(L.clsprc)

最后画出对数收益率直到 40 阶的自相关函数图。如果收益率近似不可预测，其自相关系数应大多落在置信带之内，这与有效市场假说下收益率近似为白噪声的直觉一致。

In [ ]:
%%stata
ac rr, lags(40)